# Extract Data Sources from Power BI Semantic Models

This notebook extracts and parses datasource information from Power BI semantic model partitions stored in the LineageScanner lakehouse.

The approach is based on parsing M expressions (Power Query code) to identify:
- Connector types (e.g., Sql.Database, Web.Contents, SharePoint.Contents)
- Source locations (e.g., server names, URLs, file paths)

**Reference**: [How to Discover and Document Data Sources in Power BI Semantic Models](https://community.fabric.microsoft.com/t5/Power-BI-Community-Blog/How-to-Discover-and-Document-Data-Sources-in-Power-BI-Semantic/ba-p/5125520)

## Step 1: Import Required Libraries

In [1]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 3, Finished, Available, Finished, False)

## Step 2: Read Partition Data from Lakehouse

Load the `t_dataset_partitions` table from the LineageScanner lakehouse. This table contains M expressions for each table partition in Power BI semantic models.

In [2]:
# Read the partitions table from the lakehouse
df_partitions = spark.read.table("LineageScanner.dbo.t_dataset_partitions")

# Display basic info
print(f"Total partitions: {df_partitions.count()}")
print(f"\nSample of source types:")
df_partitions.groupBy("source_type").count().show()

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 4, Finished, Available, Finished, False)

Total partitions: 47

Sample of source types:
+-----------+-----+
|source_type|count|
+-----------+-----+
|          M|   29|
| Calculated|   16|
|     Entity|    2|
+-----------+-----+



## Step 3: Define Datasource Extraction Function

Create a Python UDF to parse M expressions and extract:
- **Connector Type**: The datasource connector (e.g., `Sql.Database`, `Web.Contents`)
- **Source Location**: The primary identifier (e.g., server name, URL)

The regex pattern matches M function calls like: `Connector.Function("source", ...)`

### Extraction Patterns Supported:

The function handles these M expression patterns:

1. **Sql.Database** with server and database:
   - `Sql.Database("server", "database")`
   
2. **Database name from variable reference**:
   - `Source{[Name=Database]}`
   - `Source{[Name="DatabaseName"]}`
   
3. **Schema and Table from M patterns**:
   - `{[Schema="dbo",Item="TableName"]}`
   
4. **SQL queries with FROM clauses**:
   - `[Query="SELECT ... FROM schema.table"]`
   - `[Query="SELECT ... FROM [schema].[table]"]`

The function extracts: **connector_type**, **server**, **database**, **schema**, and **table**.

In [3]:
def extract_connector(query):
    """
    Extract the datasource connector type and source location from an M expression.
    
    Args:
        query: M expression string (Power Query code)
    
    Returns:
        dict: {
            'connector_type': str,
            'server': str,
            'database': str,
            'schema': str,
            'table': str
        }
    """
    if not query or query.strip() == "":
        return {
            'connector_type': 'Calculated',
            'server': None,
            'database': None,
            'schema': None,
            'table': None
        }
    
    result = {
        'connector_type': 'Unknown',
        'server': None,
        'database': None,
        'schema': None,
        'table': None
    }
    
    # 1. Extract connector type (Sql.Database, Sql.Databases, Web.Contents, etc.)
    connector_pattern = r'([A-Za-z0-9_]+)\.([A-Za-z0-9_]+)\('
    connector_match = re.search(connector_pattern, query, re.IGNORECASE)
    
    if connector_match:
        result['connector_type'] = f"{connector_match.group(1)}.{connector_match.group(2)}"
    else:
        result['connector_type'] = 'Unknown / Complex M'
        return result
    
    # 2. Extract server and database from Sql.Database("server", "database")
    sql_database_pattern = r'Sql\.Database\s*\(\s*"([^"]+)"\s*,\s*"([^"]+)"'
    sql_db_match = re.search(sql_database_pattern, query, re.IGNORECASE)
    
    if sql_db_match:
        result['server'] = sql_db_match.group(1)
        result['database'] = sql_db_match.group(2)
    
    # 3. Extract database from {[Name="..."]} or {[Name=Variable]}
    name_pattern = r'\{\[Name\s*=\s*"?([^"\]]+)"?\]\}'
    name_match = re.search(name_pattern, query, re.IGNORECASE)
    
    if name_match and not result['database']:
        result['database'] = name_match.group(1)
    
    # 4. Extract schema and table from [Schema="...",Item="..."]
    schema_item_pattern = r'\[Schema\s*=\s*"([^"]+)"\s*,\s*Item\s*=\s*"([^"]+)"\]'
    schema_item_match = re.search(schema_item_pattern, query, re.IGNORECASE)
    
    if schema_item_match:
        result['schema'] = schema_item_match.group(1)
        result['table'] = schema_item_match.group(2)
    
    # 5. Extract schema.table from SQL Query in [Query="SELECT ... FROM schema.table"]
    # Look for FROM clause patterns
    query_pattern = r'\[Query\s*=\s*"([^"]+)"\]'
    query_match = re.search(query_pattern, query, re.IGNORECASE | re.DOTALL)
    
    if query_match:
        sql_query = query_match.group(1)
        
        # Pattern for FROM schema.table
        from_pattern = r'(?:FROM|JOIN)\s+(?:\[)?([a-zA-Z0-9_]+)(?:\])?\.(?:\[)?([a-zA-Z0-9_]+)(?:\])?'
        from_match = re.search(from_pattern, sql_query, re.IGNORECASE)
        
        if from_match and not result['schema']:
            result['schema'] = from_match.group(1)
            result['table'] = from_match.group(2)
    
    return result


# Test the function with the provided examples
print("=" * 80)
print("Test 1: Sql.Databases with Schema/Item pattern")
print("=" * 80)
test_query_1 = '''let
    Source = Sql.Databases(SQLInstance),
    AdventureWorksDW2020 = Source{[Name=Database]}[Data],
    dbo_DimProduct = AdventureWorksDW2020{[Schema="dbo",Item="DimProduct"]}[Data]
in
    dbo_DimProduct'''

result1 = extract_connector(test_query_1)
print(f"Input 1: {test_query_1[:80]}...")
print(f"Result 1: {result1}")
print(f"Expected: database='Database', schema='dbo', table='DimProduct'")

print("\n" + "=" * 80)
print("Test 2: Sql.Database with embedded SQL Query")
print("=" * 80)
test_query_2 = '''let
    Source = Sql.Database(".", "IP", [Query="select distinct market BU,#(lf)  REGIONTITLE Region,#(lf)  MARKETDIRECTOR VP#(lf)from hr.bu"]),
    #"Renamed Columns" = Table.RenameColumns(Source, {{"BU", "BU"}, {"Region", "RegionSeq"}, {"VP", "VP"}}),
    #"Changed Type" = Table.TransformColumnTypes(#"Renamed Columns", {{"BU", type text}, {"RegionSeq", type text}, {"VP", type text}})
in
    #"Changed Type"'''

result2 = extract_connector(test_query_2)
print(f"Input 2: {test_query_2[:80]}...")
print(f"Result 2: {result2}")
print(f"Expected: server='.', database='IP', schema='hr', table='bu'")

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 5, Finished, Available, Finished, False)

Test 1: Sql.Databases with Schema/Item pattern
Input 1: let
    Source = Sql.Databases(SQLInstance),
    AdventureWorksDW2020 = Source{[...
Result 1: {'connector_type': 'Sql.Databases', 'server': None, 'database': 'Database', 'schema': 'dbo', 'table': 'DimProduct'}
Expected: database='Database', schema='dbo', table='DimProduct'

Test 2: Sql.Database with embedded SQL Query
Input 2: let
    Source = Sql.Database(".", "IP", [Query="select distinct market BU,#(lf)...
Result 2: {'connector_type': 'Sql.Database', 'server': '.', 'database': 'IP', 'schema': 'hr', 'table': 'bu'}
Expected: server='.', database='IP', schema='hr', table='bu'


## Step 4: Extract Datasources from All Partitions

Apply the extraction function to all M-based partitions and create a structured dataset with:
- Table name
- Connector type  
- Source location
- Dataset and workspace identifiers

In [4]:
# Filter for M-based partitions (exclude calculated tables, etc.)
df_m_partitions = df_partitions.filter(F.col("source_type") == "M")

# Define a UDF that returns a struct with all extracted fields
from pyspark.sql.types import StructType, StructField

@F.udf(StructType([
    StructField("connector_type", StringType(), True),
    StructField("server", StringType(), True),
    StructField("database", StringType(), True),
    StructField("schema", StringType(), True),
    StructField("table", StringType(), True)
]))
def extract_connector_struct(query):
    """Extract connector information as a struct with multiple fields"""
    if not query or query.strip() == "":
        return ("Calculated", None, None, None, None)
    
    result = {
        'connector_type': 'Unknown',
        'server': None,
        'database': None,
        'schema': None,
        'table': None
    }
    
    # 1. Extract connector type
    connector_pattern = r'([A-Za-z0-9_]+)\.([A-Za-z0-9_]+)\('
    connector_match = re.search(connector_pattern, query, re.IGNORECASE)
    
    if connector_match:
        result['connector_type'] = f"{connector_match.group(1)}.{connector_match.group(2)}"
    else:
        result['connector_type'] = 'Unknown / Complex M'
        return (result['connector_type'], None, None, None, None)
    
    # 2. Extract server and database from Sql.Database("server", "database")
    sql_database_pattern = r'Sql\.Database\s*\(\s*"([^"]+)"\s*,\s*"([^"]+)"'
    sql_db_match = re.search(sql_database_pattern, query, re.IGNORECASE)
    
    if sql_db_match:
        result['server'] = sql_db_match.group(1)
        result['database'] = sql_db_match.group(2)
    
    # 3. Extract database from {[Name="..."]} or {[Name=Variable]}
    name_pattern = r'\{\[Name\s*=\s*"?([^"\]]+)"?\]\}'
    name_match = re.search(name_pattern, query, re.IGNORECASE)
    
    if name_match and not result['database']:
        result['database'] = name_match.group(1)
    
    # 4. Extract schema and table from [Schema="...",Item="..."]
    schema_item_pattern = r'\[Schema\s*=\s*"([^"]+)"\s*,\s*Item\s*=\s*"([^"]+)"\]'
    schema_item_match = re.search(schema_item_pattern, query, re.IGNORECASE)
    
    if schema_item_match:
        result['schema'] = schema_item_match.group(1)
        result['table'] = schema_item_match.group(2)
    
    # 5. Extract schema.table from SQL Query in [Query="SELECT ... FROM schema.table"]
    query_pattern = r'\[Query\s*=\s*"([^"]+)"\]'
    query_match = re.search(query_pattern, query, re.IGNORECASE | re.DOTALL)
    
    if query_match:
        sql_query = query_match.group(1)
        
        # Pattern for FROM schema.table
        from_pattern = r'(?:FROM|JOIN)\s+(?:\[)?([a-zA-Z0-9_]+)(?:\])?\.(?:\[)?([a-zA-Z0-9_]+)(?:\])?'
        from_match = re.search(from_pattern, sql_query, re.IGNORECASE)
        
        if from_match and not result['schema']:
            result['schema'] = from_match.group(1)
            result['table'] = from_match.group(2)
    
    return (
        result['connector_type'],
        result['server'],
        result['database'],
        result['schema'],
        result['table']
    )

# Apply the extraction
df_extracted = df_m_partitions.withColumn("connector_info", extract_connector_struct(F.col("query")))

# Expand the struct into separate columns
df_datasources = df_extracted.select(
    F.col("table_name").alias("power_bi_table_name"),
    F.col("connector_info.connector_type").alias("connector"),
    F.col("connector_info.server").alias("source_server"),
    F.col("connector_info.database").alias("source_database"),
    F.col("connector_info.schema").alias("source_schema"),
    F.col("connector_info.table").alias("source_table"),
    F.col("dataset_id"),
    F.col("workspace_id")
)

# Display results
print("Extracted Datasources:")
df_datasources.show(20, truncate=False)

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 6, Finished, Available, Finished, False)

Extracted Datasources:
+--------------------+--------------+----------------------------------------------------------------------------------------+---------------+-------------+--------------------+------------------------------------+------------------------------------+
|power_bi_table_name |connector     |source_server                                                                           |source_database|source_schema|source_table        |dataset_id                          |workspace_id                        |
+--------------------+--------------+----------------------------------------------------------------------------------------+---------------+-------------+--------------------+------------------------------------+------------------------------------+
|Salesperson         |Sql.Databases |NULL                                                                                    |Database       |dbo          |DimEmployee         |80490e11-fa1f-4d7c-a5e3-fefa2d877221|2b1d454

## Step 5: Analyze Datasource Distribution

Group the results by connector type to understand which datasources are most commonly used.

In [5]:
# Group by connector type to see distribution
connector_summary = df_datasources.groupBy("connector").count().orderBy(F.desc("count"))

print("Datasource Connector Distribution:")
connector_summary.show(truncate=False)

# Show unique database sources
print("\n" + "=" * 80)
print("Unique Database Sources (Server + Database):")
print("=" * 80)
df_datasources.filter(F.col("source_database").isNotNull()) \
    .groupBy("source_server", "source_database") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(30, truncate=False)

# Show unique schema.table combinations
print("\n" + "=" * 80)
print("Unique Schema.Table References:")
print("=" * 80)
df_datasources.filter(F.col("source_schema").isNotNull()) \
    .groupBy("source_database", "source_schema", "source_table") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(30, truncate=False)

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 7, Finished, Available, Finished, False)

Datasource Connector Distribution:
+-----------------------+-----+
|connector              |count|
+-----------------------+-----+
|Sql.Database           |17   |
|Sql.Databases          |5    |
|Table.FromRows         |4    |
|Csv.Document           |1    |
|PowerPlatform.Dataflows|1    |
|Json.Document          |1    |
+-----------------------+-----+


Unique Database Sources (Server + Database):
+----------------------------------------------------------------------------------------+---------------+-----+
|source_server                                                                           |source_database|count|
+----------------------------------------------------------------------------------------+---------------+-----+
|e23bihhxygau5oihy3pjdwvgku-4orbhul6hsfedbffmt43mtrdgq.datawarehouse.fabric.microsoft.com|LH_Bronze      |11   |
|.                                                                                       |IP             |6    |
|NULL                            

## Step 6: Save Results to Lakehouse

Save the extracted datasources to a Delta table in the LineageScanner lakehouse for further analysis and reporting.

In [6]:
# Save to Delta table in the lakehouse
table_name = "t_dataset_datasources_extracted"

df_datasources.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"LineageScanner.dbo.{table_name}")

print(f"✓ Datasources saved to table: {table_name}")
print(f"Total records: {df_datasources.count()}")

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 8, Finished, Available, Finished, False)

✓ Datasources saved to table: t_dataset_datasources_extracted
Total records: 29


## Step 7: Review Complex M Expressions (Optional)

For entries marked as "Unknown / Complex M", you may want to review the original M expressions to manually identify datasources.

In [7]:
# Find entries with unknown/complex M expressions
df_complex = df_extracted.filter(
    F.col("connector_info.connector_type") == "Unknown / Complex M"
).select("table_name", "query", "dataset_id")

print(f"Tables with complex M expressions: {df_complex.count()}")

# Show a few examples (truncated for readability)
if df_complex.count() > 0:
    print("\nExamples (first 3):")
    df_complex.show(3, truncate=100, vertical=True)

# Also check for SQL-based sources that didn't extract schema/table info
print("\n" + "=" * 80)
print("SQL Sources Missing Schema/Table Info (may need manual review):")
print("=" * 80)
df_missing_detail = df_datasources.filter(
    (F.col("connector").like("%Sql%")) & 
    (F.col("source_schema").isNull() | F.col("source_table").isNull())
).select("power_bi_table_name", "connector", "source_server", "source_database", "dataset_id")

print(f"Count: {df_missing_detail.count()}")
if df_missing_detail.count() > 0:
    df_missing_detail.show(10, truncate=False)

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 9, Finished, Available, Finished, False)

Tables with complex M expressions: 0

SQL Sources Missing Schema/Table Info (may need manual review):
Count: 0


StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, -1, Finished, Available, Finished, True)

## Step 8: Track Column Transformation Lineage

This section implements **column-level lineage tracking** for Power Query M expressions. 

### Approach:
1. **Parse M expressions** into individual transformation steps
2. **Work backwards** from the final output to trace each column's history
3. **Track column name changes** through rename operations
4. **Identify affected columns** for each transformation:
   - Direct impacts (e.g., columns renamed, replaced)
   - Indirect impacts (e.g., whole-table operations like SelectRows)
   
### Supported Transformations:
- `Table.RenameColumns` - column renames
- `Table.ReplaceValue` - value replacements in specific columns
- `Table.SelectRows` - row filtering (affects entire table)
- `Table.RemoveColumns` - column removal
- `Table.SelectColumns` - column selection
- `Table.TransformColumnTypes` - type transformations
- And more...

### Output:
For each column in the final output, we'll show all transformation steps that affected it, in reverse chronological order (most recent first).


In [9]:
import re
from typing import Dict, List, Tuple, Set
from collections import defaultdict

class MExpressionParser:
    """
    Parser for Power Query M expressions to track column-level lineage.
    Works backwards from final output to trace transformation history for each column.
    """
    
    def __init__(self, m_expression: str):
        self.m_expression = m_expression
        self.steps = []
        self.column_lineage = defaultdict(list)
        
    def parse_steps(self) -> List[Dict]:
        """
        Parse M expression into individual transformation steps.
        Each step is a dictionary with: name, expression, function_used
        """
        if not self.m_expression or self.m_expression.strip() == "":
            return []
        
        # Extract the let...in block
        # Handle both simple names and #"Name with spaces"
        let_pattern = r'let\s+(.*?)\s+in\s+(#?"[^"]+"|[a-zA-Z_][a-zA-Z0-9_]*)'
        match = re.search(let_pattern, self.m_expression, re.DOTALL | re.IGNORECASE)
        
        if not match:
            return []
        
        step_definitions = match.group(1).strip()
        final_step = match.group(2).strip().strip('"').strip('#"')
        
        # Parse individual step assignments
        # Match patterns like: StepName = Expression, or #"Step Name" = Expression,
        # We need to match balanced parentheses and handle commas inside expressions
        steps = []
        
        # Split by lines and reconstruct steps
        lines = step_definitions.split('\n')
        current_step = None
        current_expression = []
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
            
            # Check if this line starts a new step (contains = near the beginning)
            # Match: name = or #"name" =
            step_start = re.match(r'^(#?"[^"]+"|[a-zA-Z_][a-zA-Z0-9_]*)\s*=\s*(.*)$', line)
            
            if step_start:
                # Save previous step if exists
                if current_step:
                    expression = ' '.join(current_expression).rstrip(',').strip()
                    function_match = re.search(r'(Table\.[a-zA-Z]+|[a-zA-Z]+\.[a-zA-Z]+)', expression)
                    function_used = function_match.group(1) if function_match else 'Unknown'
                    
                    steps.append({
                        'name': current_step,
                        'expression': expression,
                        'function': function_used
                    })
                
                # Start new step
                current_step = step_start.group(1).strip().strip('"').strip('#"')
                current_expression = [step_start.group(2)]
            else:
                # Continuation of previous step
                if current_step:
                    current_expression.append(line)
        
        # Don't forget the last step
        if current_step:
            expression = ' '.join(current_expression).rstrip(',').strip()
            function_match = re.search(r'(Table\.[a-zA-Z]+|[a-zA-Z]+\.[a-zA-Z]+)', expression)
            function_used = function_match.group(1) if function_match else 'Unknown'
            
            steps.append({
                'name': current_step,
                'expression': expression,
                'function': function_used
            })
        
        self.steps = steps
        self.final_step = final_step
        return steps
    
    def extract_column_references(self, expression: str) -> Set[str]:
        """Extract column names referenced in an expression."""
        columns = set()
        
        # Pattern 1: {"OldName", "NewName"} in RenameColumns
        rename_pattern = r'\{"([^"]+)",\s*"([^"]+)"\}'
        for match in re.finditer(rename_pattern, expression):
            columns.add(match.group(1))
            columns.add(match.group(2))
        
        # Pattern 2: {"ColumnName"} in ReplaceValue or RemoveColumns
        list_pattern = r'\{"([^"]+)"\}'
        for match in re.finditer(list_pattern, expression):
            columns.add(match.group(1))
        
        # Pattern 3: [ColumnName] in expressions
        bracket_pattern = r'\[([a-zA-Z_][a-zA-Z0-9_ ]*)\]'
        for match in re.finditer(bracket_pattern, expression):
            columns.add(match.group(1))
        
        return columns
    
    def extract_source_columns_from_expression(self, expression: str) -> Set[str]:
        """
        Extract source column names from a calculation expression.
        Used for AddColumn, DuplicateColumn, etc. to find dependencies.
        Looks for [ColumnName] patterns in the 'each' clause.
        """
        columns = set()
        
        # Look for the 'each' clause in AddColumn expressions
        # Pattern: each [Column1] & [Column2] ...
        each_pattern = r'each\s+(.+?)(?:\)|,\s*type\s|$)'
        each_match = re.search(each_pattern, expression, re.IGNORECASE | re.DOTALL)
        
        if each_match:
            each_clause = each_match.group(1)
            # Extract all [ColumnName] references
            bracket_pattern = r'\[([a-zA-Z_][a-zA-Z0-9_ ]*)\]'
            for match in re.finditer(bracket_pattern, each_clause):
                columns.add(match.group(1))
        
        return columns
    
    def analyze_step_impact(self, step: Dict) -> Dict:
        """
        Analyze how a transformation step impacts columns.
        Returns a dictionary with:
        - affected_columns: columns directly modified
        - column_mappings: old_name -> new_name for renames
        - affects_all: whether the step affects all columns
        - creates_column: name of newly created column (for AddColumn)
        """
        function = step['function']
        expression = step['expression']
        
        result = {
            'affected_columns': set(),
            'column_mappings': {},
            'affects_all': False,
            'creates_column': None
        }
        
        # Table.RenameColumns
        if 'RenameColumns' in function:
            rename_pattern = r'\{"([^"]+)",\s*"([^"]+)"\}'
            for match in re.finditer(rename_pattern, expression):
                old_name = match.group(1)
                new_name = match.group(2)
                result['affected_columns'].add(new_name)
                result['column_mappings'][new_name] = old_name
        
        # Table.AddColumn - creates a new column
        elif 'AddColumn' in function:
            # Extract new column name: Table.AddColumn(source, "NewColumnName", ...)
            add_col_pattern = r'AddColumn\([^,]+,\s*"([^"]+)"'
            match = re.search(add_col_pattern, expression)
            if match:
                new_col_name = match.group(1)
                result['creates_column'] = new_col_name
                result['affected_columns'].add(new_col_name)
        
        # Table.DuplicateColumn - creates a new column
        elif 'DuplicateColumn' in function:
            # Extract new column name: Table.DuplicateColumn(source, "SourceCol", "NewCol")
            dup_col_pattern = r'DuplicateColumn\([^,]+,\s*"[^"]+",\s*"([^"]+)"'
            match = re.search(dup_col_pattern, expression)
            if match:
                new_col_name = match.group(1)
                result['creates_column'] = new_col_name
                result['affected_columns'].add(new_col_name)
        
        # Table.ReplaceValue
        elif 'ReplaceValue' in function:
            # Extract the columns being replaced
            columns_pattern = r'Replacer\.[a-zA-Z]+,\s*\{([^}]+)\}'
            match = re.search(columns_pattern, expression)
            if match:
                cols_str = match.group(1)
                for col_match in re.finditer(r'"([^"]+)"', cols_str):
                    result['affected_columns'].add(col_match.group(1))
            else:
                # If no specific columns listed, it might affect all columns
                result['affects_all'] = True
        
        # Table.SelectRows - affects entire table (all rows, all columns)
        elif 'SelectRows' in function:
            result['affects_all'] = True
            # Extract columns referenced in the condition (for documentation)
            result['affected_columns'] = self.extract_column_references(expression)
        
        # Table.RemoveColumns
        elif 'RemoveColumns' in function:
            result['affects_all'] = True  # Affects the table structure
            # These columns are removed, so track them
            cols_pattern = r'\{([^}]+)\}'
            match = re.search(cols_pattern, expression)
            if match:
                cols_str = match.group(1)
                for col_match in re.finditer(r'"([^"]+)"', cols_str):
                    result['affected_columns'].add(col_match.group(1))
        
        # Table.SelectColumns
        elif 'SelectColumns' in function:
            result['affects_all'] = True  # Affects which columns exist
            cols_pattern = r'\{([^}]+)\}'
            match = re.search(cols_pattern, expression)
            if match:
                cols_str = match.group(1)
                for col_match in re.finditer(r'"([^"]+)"', cols_str):
                    result['affected_columns'].add(col_match.group(1))
        
        # Table.TransformColumnTypes, Table.TransformColumns
        elif 'Transform' in function:
            # Extract columns being transformed
            cols_pattern = r'\{"([^"]+)"'
            for match in re.finditer(cols_pattern, expression):
                result['affected_columns'].add(match.group(1))
        
        # Table-wide operations that affect all columns
        elif any(x in function for x in ['Table.Sort', 'Table.Distinct', 'Table.FirstN', 
                                          'Table.LastN', 'Table.Range', 'Table.Skip']):
            result['affects_all'] = True
        
        # Source operations - affect all columns
        elif any(x in function for x in ['Sql.Database', 'Sql.Databases', 'Source', 
                                          'Navigation', 'Table.', 'Data']):
            result['affects_all'] = True
        
        return result
    
    def track_column_lineage(self, target_column: str, is_dependency: bool = False, parent_column: str = None) -> List[Dict]:
        """
        Track the lineage of a specific column backwards through all transformation steps.
        Returns a list of steps that affected this column, in reverse order (newest first).
        
        Args:
            target_column: The column to track
            is_dependency: Whether this is tracking a dependency (source column used in calculation)
            parent_column: If this is a dependency, the column that depends on it
        """
        if not self.steps:
            self.parse_steps()
        
        lineage = []
        current_column_name = target_column
        column_was_created = False
        
        # Work backwards through the steps
        for step in reversed(self.steps):
            impact = self.analyze_step_impact(step)
            
            # Check if this step creates the column we're tracking
            if impact['creates_column'] == current_column_name:
                # Extract source columns from the creation expression
                source_columns = self.extract_source_columns_from_expression(step['expression'])
                
                lineage.append({
                    'step_name': step['name'],
                    'step_expression': step['expression'],
                    'function': step['function'],
                    'column_name_at_step': current_column_name,
                    'affects_all': False,
                    'column_created_here': True,
                    'is_dependency': is_dependency,
                    'parent_column': parent_column,
                    'source_columns': list(source_columns) if source_columns else None
                })
                
                # If this column was created from other columns, track those dependencies
                if source_columns and not is_dependency:
                    for source_col in source_columns:
                        dependency_lineage = self.track_column_lineage(
                            source_col, 
                            is_dependency=True, 
                            parent_column=current_column_name
                        )
                        lineage.extend(dependency_lineage)
                
                column_was_created = True
                # Stop tracking - column didn't exist before this step
                break
            
            # Check if this step affects our column
            is_affected = False
            
            # IMPORTANT: Table-wide operations affect ALL columns
            if impact['affects_all']:
                is_affected = True
            # Column-specific operations only if the column is mentioned
            elif current_column_name in impact['affected_columns']:
                is_affected = True
            
            if is_affected:
                lineage.append({
                    'step_name': step['name'],
                    'step_expression': step['expression'],
                    'function': step['function'],
                    'column_name_at_step': current_column_name,
                    'affects_all': impact['affects_all'],
                    'column_created_here': False,
                    'is_dependency': is_dependency,
                    'parent_column': parent_column,
                    'source_columns': None
                })
                
                # If column was renamed at this step, track the old name
                if current_column_name in impact['column_mappings']:
                    current_column_name = impact['column_mappings'][current_column_name]
        
        return lineage
    
    def get_all_final_columns(self) -> Set[str]:
        """Extract all column names from the final step."""
        if not self.steps:
            self.parse_steps()
        
        columns = set()
        
        # Look at the last few steps to find column names
        for step in reversed(self.steps[-3:]):
            cols = self.extract_column_references(step['expression'])
            columns.update(cols)
        
        return columns

print("✓ M Expression Parser class loaded successfully")
print("\nUsage:")
print("  parser = MExpressionParser(m_query_string)")
print("  parser.parse_steps()")
print("  lineage = parser.track_column_lineage('ColumnName')")

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 10, Finished, Available, Finished, False)

✓ M Expression Parser class loaded successfully

Usage:
  parser = MExpressionParser(m_query_string)
  parser.parse_steps()
  lineage = parser.track_column_lineage('ColumnName')


### Test with Example Query

Let's test the parser with the example query provided. We'll track the lineage of the "JobTitle" column.

In [10]:
# Test query from user
test_query = '''let
    Source = Sql.Database("e23bihhxygau5oihy3pjdwvgku-4orbhul6hsfedbffmt43mtrdgq.datawarehouse.fabric.microsoft.com", "LH_Bronze"),
    dbo_Employees = Source{[Schema="dbo",Item="Employees"]}[Data],
    #"Filtered Rows" = Table.SelectRows(dbo_Employees, each ([Group] <> "NULL")),
    #"Renamed Columns" = Table.RenameColumns(#"Filtered Rows",{{"FullName", "Full Name"}, {"JobTitle", "Job Title"}, {"FirstName", "First Name"}, {"LastName", "Last Name"}}),
    #"Replaced Value" = Table.ReplaceValue(#"Renamed Columns"," ","_",Replacer.ReplaceText,{"Job Title"}),
    #"Renamed Columns1" = Table.RenameColumns(#"Replaced Value",{{"Job Title", "JobTitle"}})
in
    #"Renamed Columns1"'''

# Parse the query
parser = MExpressionParser(test_query)
steps = parser.parse_steps()

print("=" * 100)
print("PARSED TRANSFORMATION STEPS")
print("=" * 100)
for i, step in enumerate(steps, 1):
    print(f"\n{i}. Step: {step['name']}")
    print(f"   Function: {step['function']}")
    print(f"   Expression: {step['expression'][:100]}...")

# Track lineage for JobTitle column
print("\n" + "=" * 100)
print("COLUMN LINEAGE FOR: JobTitle")
print("=" * 100)

lineage = parser.track_column_lineage("JobTitle")

for i, step_info in enumerate(lineage, 1):
    print(f"\n{i}. Step: {step_info['step_name']}")
    print(f"   Function: {step_info['function']}")
    print(f"   Column name at this step: {step_info['column_name_at_step']}")
    print(f"   Affects entire table: {step_info['affects_all']}")
    print(f"   Expression: {step_info['step_expression']}")

print("\n" + "=" * 100)
print("✓ Column lineage tracking completed!")
print("=" * 100)

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 11, Finished, Available, Finished, False)

PARSED TRANSFORMATION STEPS

1. Step: Source
   Function: Sql.Database
   Expression: Sql.Database("e23bihhxygau5oihy3pjdwvgku-4orbhul6hsfedbffmt43mtrdgq.datawarehouse.fabric.microsoft.c...

2. Step: dbo_Employees
   Function: Unknown
   Expression: Source{[Schema="dbo",Item="Employees"]}[Data]...

3. Step: Filtered Rows
   Function: Table.SelectRows
   Expression: Table.SelectRows(dbo_Employees, each ([Group] <> "NULL"))...

4. Step: Renamed Columns
   Function: Table.RenameColumns
   Expression: Table.RenameColumns(#"Filtered Rows",{{"FullName", "Full Name"}, {"JobTitle", "Job Title"}, {"FirstN...

5. Step: Replaced Value
   Function: Table.ReplaceValue
   Expression: Table.ReplaceValue(#"Renamed Columns"," ","_",Replacer.ReplaceText,{"Job Title"})...

6. Step: Renamed Columns1
   Function: Table.RenameColumns
   Expression: Table.RenameColumns(#"Replaced Value",{{"Job Title", "JobTitle"}})...

COLUMN LINEAGE FOR: JobTitle

1. Step: Renamed Columns1
   Function: Table.RenameColumns
 

### Test with Extended Example

Let's test with a more complex example that includes:
1. **Table.AddColumn** - to verify new columns are tracked
2. **Columns not explicitly mentioned** - to verify table-wide transformations apply to all columns

In [11]:
# Extended test query with AddColumn and implicit columns
test_query_extended = '''let
    Source = Sql.Database("server.com", "MyDB"),
    dbo_Employees = Source{[Schema="dbo",Item="Employees"]}[Data],
    #"Filtered Rows" = Table.SelectRows(dbo_Employees, each ([Status] = "Active")),
    #"Added Custom" = Table.AddColumn(#"Filtered Rows", "FullName", each [FirstName] & " " & [LastName]),
    #"Renamed Columns" = Table.RenameColumns(#"Added Custom",{{"FullName", "Full Name"}})
in
    #"Renamed Columns"'''

# Parse the extended query
parser_ext = MExpressionParser(test_query_extended)
steps_ext = parser_ext.parse_steps()

print("=" * 100)
print("TEST 1: Newly Created Column (AddColumn)")
print("=" * 100)
print("\nTracking lineage for 'Full Name' (created via AddColumn):\n")

lineage_fullname = parser_ext.track_column_lineage("Full Name")

for i, step_info in enumerate(lineage_fullname, 1):
    print(f"{i}. Step: {step_info['step_name']}")
    print(f"   Function: {step_info['function']}")
    print(f"   Column created here: {step_info.get('column_created_here', False)}")
    print(f"   Affects entire table: {step_info['affects_all']}")
    print()

print("=" * 100)
print("TEST 2: Column Not Explicitly Mentioned")
print("=" * 100)
print("\nTracking lineage for 'FirstName' (never explicitly transformed, just passed through):\n")

lineage_firstname = parser_ext.track_column_lineage("FirstName")

for i, step_info in enumerate(lineage_firstname, 1):
    print(f"{i}. Step: {step_info['step_name']}")
    print(f"   Function: {step_info['function']}")
    print(f"   Column name at step: {step_info['column_name_at_step']}")
    print(f"   Affects entire table: {step_info['affects_all']}")
    print()

print("=" * 100)
print("VERIFICATION")
print("=" * 100)
print("✓ Test 1: 'Full Name' should show it was CREATED in 'Added Custom' step")
print("✓ Test 2: 'FirstName' should show ALL table-wide operations (Filtered Rows, Source, etc.)")
print("=" * 100)

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 12, Finished, Available, Finished, False)

TEST 1: Newly Created Column (AddColumn)

Tracking lineage for 'Full Name' (created via AddColumn):

1. Step: Renamed Columns
   Function: Table.RenameColumns
   Column created here: False
   Affects entire table: False

2. Step: Added Custom
   Function: Table.AddColumn
   Column created here: True
   Affects entire table: False

3. Step: Filtered Rows
   Function: Table.SelectRows
   Column created here: False
   Affects entire table: True

4. Step: Source
   Function: Sql.Database
   Column created here: False
   Affects entire table: True

5. Step: Filtered Rows
   Function: Table.SelectRows
   Column created here: False
   Affects entire table: True

6. Step: Source
   Function: Sql.Database
   Column created here: False
   Affects entire table: True

TEST 2: Column Not Explicitly Mentioned

Tracking lineage for 'FirstName' (never explicitly transformed, just passed through):

1. Step: Filtered Rows
   Function: Table.SelectRows
   Column name at step: FirstName
   Affects entire 

### Test Column Dependencies

Let's test tracking dependencies - when a column is created from other columns (like AddColumn), we should track the lineage of those source columns too.

In [12]:
# Test query with column dependencies (similar to user's example)
test_query_dependencies = '''let
    Source = Sql.Database("server.com", "MyDB"),
    dbo_Employees = Source{[Schema="dbo",Item="Employees"]}[Data],
    #"Filtered Rows" = Table.SelectRows(dbo_Employees, each ([Status] = "Active")),
    #"Renamed Columns" = Table.RenameColumns(#"Filtered Rows",{{"Country", "Country_Renamed"}}),
    #"Renamed Columns1" = Table.RenameColumns(#"Renamed Columns",{{"Group", "Group_Renamed"}}),
    #"Added Country-Group" = Table.AddColumn(#"Renamed Columns1", "Country-Group", each [Country_Renamed] & ", " & [Group_Renamed])
in
    #"Added Country-Group"'''

# Parse the query
parser_dep = MExpressionParser(test_query_dependencies)
steps_dep = parser_dep.parse_steps()

print("=" * 100)
print("TEST 3: Column Dependencies - Tracking Source Columns")
print("=" * 100)
print("\nTracking lineage for 'Country-Group' (created from [Country_Renamed] & [Group_Renamed]):\n")

lineage_country_group = parser_dep.track_column_lineage("Country-Group")

# Separate direct steps from dependency steps
direct_steps = [s for s in lineage_country_group if not s.get('is_dependency', False)]
dependency_steps = [s for s in lineage_country_group if s.get('is_dependency', False)]

print("DIRECT TRANSFORMATION STEPS FOR 'Country-Group':")
print("=" * 100)
for i, step_info in enumerate(direct_steps, 1):
    print(f"\n{i}. Step: {step_info['step_name']}")
    print(f"   Function: {step_info['function']}")
    print(f"   Column created here: {step_info.get('column_created_here', False)}")
    if step_info.get('source_columns'):
        print(f"   ✓ Created from source columns: {step_info.get('source_columns')}")
    print(f"   Affects entire table: {step_info['affects_all']}")

print("\n" + "=" * 100)
print("DEPENDENCY STEPS (transformations of source columns):")
print("=" * 100)

# Group by parent column
from itertools import groupby
dependency_steps_sorted = sorted(dependency_steps, key=lambda x: (x.get('parent_column', ''), x.get('column_name_at_step', '')))

for parent_col, group in groupby(dependency_steps_sorted, key=lambda x: x.get('parent_column')):
    print(f"\n▶ Source column for '{parent_col}':")
    for i, step_info in enumerate(group, 1):
        print(f"  {i}. Step: {step_info['step_name']}")
        print(f"     Column: {step_info['column_name_at_step']}")
        print(f"     Function: {step_info['function']}")
        print(f"     Affects entire table: {step_info['affects_all']}")

print("\n" + "=" * 100)
print("VERIFICATION")
print("=" * 100)
print("✓ 'Country-Group' should show it was CREATED from [Country_Renamed] and [Group_Renamed]")
print("✓ Dependency steps should show:")
print("  - Transformations for Country_Renamed (renamed from Country)")
print("  - Transformations for Group_Renamed (renamed from Group)")
print("  - Table-wide operations (Filtered Rows, Source) for both")
print("=" * 100)

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 13, Finished, Available, Finished, False)

TEST 3: Column Dependencies - Tracking Source Columns

Tracking lineage for 'Country-Group' (created from [Country_Renamed] & [Group_Renamed]):

DIRECT TRANSFORMATION STEPS FOR 'Country-Group':

1. Step: Added Country-Group
   Function: Table.AddColumn
   Column created here: True
   ✓ Created from source columns: ['Group_Renamed', 'Country_Renamed']
   Affects entire table: False

DEPENDENCY STEPS (transformations of source columns):

▶ Source column for 'Country-Group':
  1. Step: Filtered Rows
     Column: Country
     Function: Table.SelectRows
     Affects entire table: True
  2. Step: Source
     Column: Country
     Function: Sql.Database
     Affects entire table: True
  3. Step: Renamed Columns
     Column: Country_Renamed
     Function: Table.RenameColumns
     Affects entire table: False
  4. Step: Filtered Rows
     Column: Group
     Function: Table.SelectRows
     Affects entire table: True
  5. Step: Source
     Column: Group
     Function: Sql.Database
     Affects enti

### Apply Column Lineage Tracking to All Partitions

Now let's apply the lineage tracking to all M-based partitions in our dataset and create a structured table with column transformation history.

In [13]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, ArrayType

# Filter to M-based partitions only
df_m_partitions = df_partitions.filter(F.col("source_type") == "M")

# Load t_dataset_columns to get the actual final columns for each table
df_dataset_columns = spark.read.table("LineageScanner.dbo.t_dataset_columns")

# Filter to only data columns (not calculated columns)
df_data_columns = df_dataset_columns.filter(F.col("type") != "Calculated")

print(f"Total columns in dataset: {df_dataset_columns.count()}")
print(f"Data columns (non-calculated): {df_data_columns.count()}")
print(f"Calculated columns: {df_dataset_columns.filter(F.col('type') == 'Calculated').count()}")

# Show sample
print("\nSample of data columns:")
df_data_columns.select("table_name", "column_name", "type", "dataset_id").show(5, truncate=False)

def extract_column_lineage_for_query(query_text: str, table_name: str, dataset_id: str, workspace_id: str, column_names: list):
    """
    Extract column lineage for a single M query with actual final columns from the semantic model.
    
    Args:
        query_text: M expression string
        table_name: Name of the Power BI table
        dataset_id: Dataset ID
        workspace_id: Workspace ID
        column_names: List of actual column names in the final table (from t_dataset_columns)
    
    Returns a list of dictionaries with lineage information.
    """
    if not query_text or query_text.strip() == "":
        return []
    
    if not column_names:
        return []
    
    try:
        parser = MExpressionParser(query_text)
        steps = parser.parse_steps()
        
        if not steps:
            return []
        
        lineage_records = []
        
        # Track lineage for each actual column in the final table
        for column in column_names:
            lineage = parser.track_column_lineage(column)
            
            # Create a record for each transformation step affecting this column
            for step_order, step_info in enumerate(lineage, 1):
                lineage_records.append({
                    'power_bi_table_name': table_name,
                    'dataset_id': dataset_id,
                    'workspace_id': workspace_id,
                    'final_column_name': column,
                    'step_order': step_order,  # 1 = most recent, higher = earlier in process
                    'step_name': step_info['step_name'],
                    'transformation_function': step_info['function'],
                    'column_name_at_step': step_info['column_name_at_step'],
                    'affects_entire_table': step_info['affects_all'],
                    'column_created_here': step_info.get('column_created_here', False),
                    'is_dependency': step_info.get('is_dependency', False),
                    'parent_column': step_info.get('parent_column'),
                    'source_columns': str(step_info.get('source_columns')) if step_info.get('source_columns') else None,
                    'step_expression': step_info['step_expression']
                })
        
        return lineage_records
    
    except Exception as e:
        # Log error but continue processing
        print(f"Error processing query for table {table_name}: {str(e)}")
        return []

# Join partitions with columns to get actual final columns for each table
df_partition_columns = df_m_partitions.alias("p").join(
    df_data_columns.alias("c"),
    (F.col("p.table_name") == F.col("c.table_name")) & 
    (F.col("p.dataset_id") == F.col("c.dataset_id")),
    "left"
).select(
    F.col("p.table_name"),
    F.col("p.query"),
    F.col("p.dataset_id"),
    F.col("p.workspace_id"),
    F.col("c.column_name")
)

# Group by partition to get list of columns for each table
df_partitions_with_columns = df_partition_columns.groupBy(
    "table_name", "query", "dataset_id", "workspace_id"
).agg(
    F.collect_list("column_name").alias("column_names")
)

print(f"\n✓ Joined partitions with actual columns")
print(f"Total partitions with column information: {df_partitions_with_columns.count()}")

# Test with a small sample first
print("\nTesting with a sample of 5 partitions...")
df_sample = df_partitions_with_columns.limit(5).collect()

sample_lineage = []
for row in df_sample:
    # Filter out None values from column list
    columns = [col for col in row['column_names'] if col is not None]
    
    lineage = extract_column_lineage_for_query(
        row['query'],
        row['table_name'],
        row['dataset_id'],
        row['workspace_id'],
        columns
    )
    sample_lineage.extend(lineage)

print(f"\n✓ Processed {len(df_sample)} partitions")
print(f"✓ Generated {len(sample_lineage)} lineage records")

# Show sample results
if sample_lineage:
    print("\nSample lineage records:")
    for i, record in enumerate(sample_lineage[:5], 1):
        print(f"\n{i}. Table: {record['power_bi_table_name']}")
        print(f"   Column: {record['final_column_name']}")
        print(f"   Step: {record['step_name']}")
        print(f"   Function: {record['transformation_function']}")
        print(f"   Column at step: {record['column_name_at_step']}")
        print(f"   Affects entire table: {record['affects_entire_table']}")
else:
    print("\n⚠️ No lineage records generated.")

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 14, Finished, Available, Finished, False)

Total columns in dataset: 246
Data columns (non-calculated): 235
Calculated columns: 11

Sample of data columns:
+----------+------------+----+------------------------------------+
|table_name|column_name |type|dataset_id                          |
+----------+------------+----+------------------------------------+
|Sales     |Unit Price  |Data|80490e11-fa1f-4d7c-a5e3-fefa2d877221|
|Sales     |Sales       |Data|80490e11-fa1f-4d7c-a5e3-fefa2d877221|
|Sales     |Cost        |Data|80490e11-fa1f-4d7c-a5e3-fefa2d877221|
|Targets   |EmployeeID  |Data|80490e11-fa1f-4d7c-a5e3-fefa2d877221|
|Targets   |TargetAmount|Data|80490e11-fa1f-4d7c-a5e3-fefa2d877221|
+----------+------------+----+------------------------------------+
only showing top 5 rows


✓ Joined partitions with actual columns
Total partitions with column information: 29

Testing with a sample of 5 partitions...

✓ Processed 5 partitions
✓ Generated 177 lineage records

Sample lineage records:

1. Table: Salesperson
   Column: Sales

### Process All Partitions and Create Lineage DataFrame

Now let's process all M-based partitions and create a comprehensive lineage dataset.

In [14]:
# Process all M-based partitions with actual columns
print("Processing all M-based partitions...")
print(f"Total partitions to process: {df_partitions_with_columns.count()}")

all_partitions = df_partitions_with_columns.collect()
all_lineage_records = []

processed_count = 0
error_count = 0

for row in all_partitions:
    try:
        # Filter out None values from column list
        columns = [col for col in row['column_names'] if col is not None]
        
        if not columns:
            # Skip partitions with no columns (shouldn't happen, but just in case)
            print(f"  Skipping {row['table_name']} - no columns found")
            continue
        
        lineage = extract_column_lineage_for_query(
            row['query'],
            row['table_name'],
            row['dataset_id'],
            row['workspace_id'],
            columns
        )
        all_lineage_records.extend(lineage)
        processed_count += 1
        
        # Progress indicator
        if processed_count % 50 == 0:
            print(f"  Processed {processed_count}/{len(all_partitions)} partitions...")
    
    except Exception as e:
        error_count += 1
        print(f"  Error processing partition for table {row['table_name']}: {str(e)}")

print(f"\n✓ Processing complete!")
print(f"  Successfully processed: {processed_count} partitions")
print(f"  Errors: {error_count}")
print(f"  Total lineage records generated: {len(all_lineage_records)}")

# Create Spark DataFrame from lineage records
if all_lineage_records:
    # Define schema
    lineage_schema = StructType([
        StructField("power_bi_table_name", StringType(), True),
        StructField("dataset_id", StringType(), True),
        StructField("workspace_id", StringType(), True),
        StructField("final_column_name", StringType(), True),
        StructField("step_order", IntegerType(), True),
        StructField("step_name", StringType(), True),
        StructField("transformation_function", StringType(), True),
        StructField("column_name_at_step", StringType(), True),
        StructField("affects_entire_table", BooleanType(), True),
        StructField("column_created_here", BooleanType(), True),
        StructField("is_dependency", BooleanType(), True),
        StructField("parent_column", StringType(), True),
        StructField("source_columns", StringType(), True),
        StructField("step_expression", StringType(), True)
    ])
    
    # Convert to DataFrame
    df_column_lineage = spark.createDataFrame(all_lineage_records, schema=lineage_schema)
    
    print("\n" + "=" * 100)
    print("COLUMN LINEAGE DATAFRAME CREATED")
    print("=" * 100)
    print(f"Total rows: {df_column_lineage.count()}")
    print(f"\nSample records:")
    df_column_lineage.show(10, truncate=False)
else:
    print("\n⚠️ No lineage records generated. Check if M expressions are being parsed correctly.")

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 15, Finished, Available, Finished, False)

Processing all M-based partitions...
Total partitions to process: 29

✓ Processing complete!
  Successfully processed: 29 partitions
  Errors: 0
  Total lineage records generated: 471

COLUMN LINEAGE DATAFRAME CREATED
Total rows: 471

Sample records:
+-------------------+------------------------------------+------------------------------------+-----------------+----------+---------------------+-----------------------+-------------------+--------------------+-------------------+-------------+-------------+--------------+--------------------------------------------------------------------------------------------------------------------------------------------+
|power_bi_table_name|dataset_id                          |workspace_id                        |final_column_name|step_order|step_name            |transformation_function|column_name_at_step|affects_entire_table|column_created_here|is_dependency|parent_column|source_columns|step_expression                                            

### Query Column Lineage for Specific Columns

Now we can query the lineage for any specific column to see all transformation steps that affected it.

In [15]:
# Example: Show transformation history for a specific column
# First, let's see what columns we have
print("Sample of columns in the lineage dataset:")
df_column_lineage.select("power_bi_table_name", "final_column_name") \
    .distinct() \
    .orderBy("power_bi_table_name", "final_column_name") \
    .show(20, truncate=False)

# Now pick a column and show its full transformation history
# Let's find a column with multiple transformation steps
columns_with_multiple_steps = df_column_lineage.groupBy("power_bi_table_name", "final_column_name") \
    .agg(F.count("*").alias("step_count")) \
    .filter(F.col("step_count") > 2) \
    .orderBy(F.desc("step_count"))

print("\n" + "=" * 100)
print("Columns with Multiple Transformation Steps:")
print("=" * 100)
columns_with_multiple_steps.show(10, truncate=False)

# Show detailed lineage for one example column
if columns_with_multiple_steps.count() > 0:
    example_row = columns_with_multiple_steps.first()
    example_table = example_row['power_bi_table_name']
    example_column = example_row['final_column_name']
    
    print(f"\n" + "=" * 100)
    print(f"DETAILED TRANSFORMATION LINEAGE FOR: {example_table}.{example_column}")
    print("=" * 100)
    
    lineage_for_column = df_column_lineage.filter(
        (F.col("power_bi_table_name") == example_table) & 
        (F.col("final_column_name") == example_column)
    ).orderBy("step_order")
    
    lineage_rows = lineage_for_column.collect()
    
    for i, row in enumerate(lineage_rows, 1):
        print(f"\n{i}. Step: {row['step_name']}")
        print(f"   Function: {row['transformation_function']}")
        print(f"   Column name at this step: {row['column_name_at_step']}")
        print(f"   Affects entire table: {row['affects_entire_table']}")
        print(f"   Expression: {row['step_expression'][:150]}...")
    
    print("\n" + "=" * 100)
else:
    print("\n⚠️ No columns found with multiple transformation steps in the sample.")

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 16, Finished, Available, Finished, False)

Sample of columns in the lineage dataset:
+--------------------+-----------------+
|power_bi_table_name |final_column_name|
+--------------------+-----------------+
|AgeGroup            |AgeGroup         |
|AgeGroup            |AgeGroupID       |
|BU                  |BU               |
|BU                  |RegionSeq        |
|BU                  |VP               |
|CatFacts            |data             |
|CatFactsFromDataflow|data             |
|Customers           |CustomerID       |
|Customers           |DateInserted     |
|Customers           |FirstName        |
|Customers           |FullName         |
|Customers           |LastName         |
|Customers           |SourceFilename   |
|Date                |Date             |
|Date                |Day              |
|Date                |Month            |
|Date                |MonthEndDate     |
|Date                |MonthNumber      |
|Date                |MonthStartDate   |
|Date                |Period           |
+--------------

### Save Column Lineage to Lakehouse

Save the column lineage data to a Delta table for further analysis and reporting.

In [16]:
# Save column lineage to Delta table
if 'df_column_lineage' in locals() and df_column_lineage.count() > 0:
    table_name = "t_dataset_column_lineage"
    
    df_column_lineage.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"LineageScanner.dbo.{table_name}")
    
    print("=" * 100)
    print(f"✓ Column lineage saved to table: LineageScanner.dbo.{table_name}")
    print("=" * 100)
    print(f"Total records: {df_column_lineage.count()}")
    
    # Summary statistics
    print("\n" + "=" * 100)
    print("SUMMARY STATISTICS")
    print("=" * 100)
    
    print("\n1. Transformation Functions Used:")
    df_column_lineage.groupBy("transformation_function") \
        .count() \
        .orderBy(F.desc("count")) \
        .show(truncate=False)
    
    print("\n2. Tables with Most Transformation Steps:")
    df_column_lineage.groupBy("power_bi_table_name") \
        .agg(
            F.countDistinct("final_column_name").alias("unique_columns"),
            F.count("*").alias("total_steps")
        ) \
        .orderBy(F.desc("total_steps")) \
        .show(10, truncate=False)
    
    print("\n3. Columns with Most Complex Lineage (Most Steps):")
    df_column_lineage.groupBy("power_bi_table_name", "final_column_name") \
        .agg(F.count("*").alias("transformation_steps")) \
        .orderBy(F.desc("transformation_steps")) \
        .show(10, truncate=False)
    
    print("\n" + "=" * 100)
    print("✓ Column lineage analysis complete!")
    print("=" * 100)
else:
    print("⚠️ No lineage data to save. The df_column_lineage DataFrame was not created or is empty.")

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 17, Finished, Available, Finished, False)

✓ Column lineage saved to table: LineageScanner.dbo.t_dataset_column_lineage
Total records: 471

SUMMARY STATISTICS

1. Transformation Functions Used:
+--------------------------+-----+
|transformation_function   |count|
+--------------------------+-----+
|Sql.Database              |146  |
|Table.RenameColumns       |62   |
|Table.SelectColumns       |48   |
|Table.TransformColumnTypes|48   |
|Table.SelectRows          |36   |
|Sql.Databases             |35   |
|Table.ExpandRecordColumn  |34   |
|Table.NestedJoin          |14   |
|Table.ExpandTableColumn   |14   |
|Table.FromRows            |8    |
|Table.CombineColumns      |5    |
|Table.AddColumn           |4    |
|Table.RemoveColumns       |4    |
|Table.ReplaceValue        |3    |
|Table.UnpivotOtherColumns |3    |
|Table.PromoteHeaders      |3    |
|Table.TransformColumns    |1    |
|PowerPlatform.Dataflows   |1    |
|Table.FromRecords         |1    |
|Table.ExpandListColumn    |1    |
+--------------------------+-----+


2. Tabl

### Verify Dependency Tracking

Let's verify that the dependency tracking is working by finding columns that were created from other columns and checking their lineage.

In [17]:
# Load the saved lineage table
df_lineage = spark.read.table("LineageScanner.dbo.t_column_lineage")

print("=" * 100)
print("COLUMN LINEAGE SUMMARY")
print("=" * 100)
print(f"Total lineage records: {df_lineage.count()}")
print(f"Records with dependencies: {df_lineage.filter(F.col('is_dependency') == True).count()}")
print(f"Records without dependencies: {df_lineage.filter(F.col('is_dependency') == False).count()}")

# Find columns that were created from other columns
created_columns = df_lineage.filter(
    (F.col("column_created_here") == True) & 
    (F.col("is_dependency") == False) &
    (F.col("source_columns").isNotNull())
).select("power_bi_table_name", "final_column_name", "source_columns").distinct()

print(f"\nColumns created from other columns: {created_columns.count()}")
print("\nSample of created columns with their source columns:")
created_columns.show(10, truncate=False)

# Let's look at the lineage for one example column (if any exist)
if created_columns.count() > 0:
    example_row = created_columns.first()
    example_table = example_row['power_bi_table_name']
    example_column = example_row['final_column_name']
    
    print("\n" + "=" * 100)
    print(f"DETAILED LINEAGE FOR: {example_table}.{example_column}")
    print("=" * 100)
    
    # Get all lineage for this column (both direct and dependencies)
    lineage_records = df_lineage.filter(
        (F.col("power_bi_table_name") == example_table) &
        (F.col("final_column_name") == example_column)
    ).orderBy("is_dependency", "step_order").collect()
    
    direct_steps = [r for r in lineage_records if not r['is_dependency']]
    dependency_steps = [r for r in lineage_records if r['is_dependency']]
    
    print(f"\nDIRECT TRANSFORMATION STEPS: {len(direct_steps)}")
    for i, row in enumerate(direct_steps, 1):
        print(f"\n{i}. Step: {row['step_name']}")
        print(f"   Function: {row['transformation_function']}")
        print(f"   Column at step: {row['column_name_at_step']}")
        print(f"   Created here: {row['column_created_here']}")
        if row['source_columns']:
            print(f"   ✓ Source columns: {row['source_columns']}")
    
    print(f"\n\nDEPENDENCY STEPS (source column transformations): {len(dependency_steps)}")
    
    # Group by parent_column
    from collections import defaultdict
    deps_by_parent = defaultdict(list)
    for row in dependency_steps:
        deps_by_parent[row['parent_column']].append(row)
    
    for parent_col, steps in deps_by_parent.items():
        print(f"\n▶ Transformations for source column (dependency of '{parent_col}'):")
        for i, row in enumerate(steps, 1):
            print(f"  {i}. Step: {row['step_name']}")
            print(f"     Column: {row['column_name_at_step']}")
            print(f"     Function: {row['transformation_function']}")
    
    print("\n" + "=" * 100)
else:
    print("\n⚠️ No columns created from other columns found in the dataset.")

StatementMeta(, 88f5ab0c-2a05-419e-b7d5-4a29c3b9b9c4, 18, Finished, Available, Finished, False)

COLUMN LINEAGE SUMMARY
Total lineage records: 471
Records with dependencies: 25
Records without dependencies: 446

Columns created from other columns: 4

Sample of created columns with their source columns:
+-------------------+-----------------+-----------------------------------------------------+
|power_bi_table_name|final_column_name|source_columns                                       |
+-------------------+-----------------+-----------------------------------------------------+
|Products           |StandardMargin   |['ListPrice', 'StandardCost']                        |
|Targets            |TargetMonth      |['MonthNumber']                                      |
|Sales              |Cost             |['OrderQuantity', 'StandardCost', 'TotalProductCost']|
|Employees          |Country-Group    |['Group', 'Country']                                 |
+-------------------+-----------------+-----------------------------------------------------+


DETAILED LINEAGE FOR: Products.Standard